## This notebook is trying to register a low res DESI and high RES Xenium OMI TIFF images together

In [2]:
import spatialdata as sd
from spatialdata_io import xenium
from pathlib import Path
from spatialdata.models import Image2DModel
from spatialdata.transformations import Scale
import tifffile
import napari

/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instea

In [3]:
user_home = Path.home()

# Define the specific project folder
project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "xenium"
msi_project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "msi"

raw_data_path = project_dir / "raw" / "output-XETG00169__0055588__55588_region_4__20250418__182706"
desi_img_data_path = msi_project_dir/ "raw"/ "3D_DESI_F5_Pos mode_40um_F5_5pos_40um_Features110425.ome.tif"

In [4]:
raw_data_path

PosixPath('/Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/raw/output-XETG00169__0055588__55588_region_4__20250418__182706')

In [5]:
sdata = xenium(raw_data_path)

INFO     reading                                                                                                   
         /Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/raw/output-XETG00169__0055588__5
         5588_region_4__20250418__182706/cell_feature_matrix.h5                                                    


/var/folders/w5/x66nm95j3cx1k13smkb3sky00000gn/T/ipykernel_34528/1396477876.py:1: DeprecationWarning: The default value of `cells_as_circles` will change to `False` in the next release. Please pass `True` explicitly to maintain the current behavior.
  sdata = xenium(raw_data_path)


/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/tifffile/tifffile.py:8797: UserWarning: <tifffile.TiffPage 0 @16> reading array from closed file
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/tifffile/tifffile.py:8797: UserWarning: <tifffile.TiffPage 0 @16> reading array from closed file
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/tifffile/tifffile.py:8797: UserWarning: <tifffile.TiffPage 0 @16> reading array from closed file
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/tifffile/tifffile.py:8797: UserWarning: <tifffile.TiffPage 0 @16> reading array from closed file
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/tifffile/tifffile.py:8797: UserWarning: <tifffile.TiffPage 0 @16> reading array from closed file
  warnings.warn(


In [6]:
sdata

SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 20495, 19973), (5, 10247, 9986), (5, 5123, 4993), (5, 2561, 2496), (5, 1280, 1248)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
│     └── 'nucleus_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (52665, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (52665, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (48131, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (52665, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), cell_circles (Shapes), nucleus_boundaries (Shapes)

In [7]:
desi_img = tifffile.imread(desi_img_data_path)

In [8]:
desi_img.shape

(88, 107, 102)

In [9]:
desi_img[87]

array([[ 59,  54,  22, ...,  12,  45,  74],
       [ 70, 180,   4, ...,   5,  86,  33],
       [ 82,  52, 142, ..., 437, 298,  20],
       ...,
       [ 11,  43, 140, ..., 245, 228, 475],
       [100,  32,  42, ..., 171,  92,  65],
       [105,  28,  75, ..., 208, 342, 356]],
      shape=(107, 102), dtype=uint16)

In [10]:
print("\n\n=== Finding DESI Channels ===")
print("We need to find channels for m/z: 616.25, 772.57, 893.75")
print("\nLet's look at some channels to identify patterns...")

# Quick visualization of several channels to find the right ones
viewer = napari.Viewer()

# Add a few DESI channels to explore
# Start with some spread across the range
test_channels = [3,17,23,25]

for i in test_channels:
    viewer.add_image(
        desi_img[i], 
        name=f"DESI_channel_{i}",
        visible=(i == 0),  # Only first one visible by default
        colormap='viridis'
    )

print(f"\nOpened napari with channels: {test_channels}")
print("Toggle through channels to find ones with clear tissue structure")
print("Note which channel numbers show the morphology you saw in QuPath")

napari.run()



=== Finding DESI Channels ===
We need to find channels for m/z: 616.25, 772.57, 893.75

Let's look at some channels to identify patterns...

Opened napari with channels: [3, 17, 23, 25]
Toggle through channels to find ones with clear tissue structure
Note which channel numbers show the morphology you saw in QuPath


In [11]:
channel_indices = [3,17,23,25]
desi_selected = desi_img[channel_indices, :, :] 

In [12]:
# Create SpatialData image with proper coordinate system
desi_spatial = Image2DModel.parse(
    desi_selected,
    dims=("c", "y", "x"),
    transformations={
        "global": Scale(
            [40.0, 40.0],  # 40 μm per pixel
            axes=("y", "x")
        )
    },
    c_coords=["mz_204.13", "heme_B_616.25", "PE_PS_731.65", "PE_PS_734.60"]
)

In [13]:
# Add to spatialdata object
sdata.images["desi"] = desi_spatial

In [14]:
print("\nDESI image added to SpatialData object!")
print(f"Available images now: {list(sdata.images.keys())}")


DESI image added to SpatialData object!
Available images now: ['morphology_focus', 'desi']


In [22]:
sdata.images['desi']

<xarray.DataArray 'image' (c: 4, y: 107, x: 102)> Size: 87kB
dask.array<array, shape=(4, 107, 102), dtype=uint16, chunksize=(4, 107, 102), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U13 208B 'mz_204.13' 'heme_B_616.25' ... 'PE_PS_734.60'
  * y        (y) float64 856B 0.5 1.5 2.5 3.5 4.5 ... 103.5 104.5 105.5 106.5
  * x        (x) float64 816B 0.5 1.5 2.5 3.5 4.5 ... 97.5 98.5 99.5 100.5 101.5
Attributes:
    transform:  {'global': Scale (y, x)\n    [40. 40.]}

### Below is the commented code to attempt and write the spatial data object in a .zarr format for faster computation

In [15]:
# # Workaround: spatialdata 0.4.0 bug — dask's to_parquet() tries to JSON-serialize
# # the .attrs on the points DataFrame, which contains non-serializable Scale/Affine
# # transformation objects. Monkey-patch write_points to strip attrs only around the
# # to_parquet() call, so _get_transformations() still works normally.
# import spatialdata._io.io_points as _io_points
# _original_write_points = _io_points.write_points

# def _patched_write_points(points, group, name, group_type="ngff:points", format=_io_points.CurrentPointsFormat()):
#     saved = points.attrs.copy()
#     original_to_parquet = points.to_parquet

#     def safe_to_parquet(*args, **kwargs):
#         points.attrs.clear()
#         points.attrs["spatialdata_attrs"] = saved.get("spatialdata_attrs", {})
#         try:
#             return original_to_parquet(*args, **kwargs)
#         finally:
#             points.attrs.clear()
#             points.attrs.update(saved)

#     points.to_parquet = safe_to_parquet
#     try:
#         _original_write_points(points, group, name, group_type, format)
#     finally:
#         points.to_parquet = original_to_parquet

# _io_points.write_points = _patched_write_points

# output_zarr_path = project_dir / "processed" / "desi_xenium_register.zarr"
# sdata.write(output_zarr_path, overwrite=True)

### Using Napari-Spatial Data to visualize

In [16]:
from napari_spatialdata import Interactive

In [23]:
# interactive = Interactive(sdata)
# interactive.run()

In [28]:
from spatialdata.models import get_channel_names

In [70]:
# 1. Create Xenium reference - combine vessel + membrane markers
morph = sdata.images['morphology_focus']

channel_names = get_channel_names(morph)
channel_names

[np.str_('DAPI'),
 np.str_('ATP1A1/CD45/E-Cadherin'),
 np.str_('18S'),
 np.str_('AlphaSMA/Vimentin'),
 np.str_('dummy')]

In [71]:
# The full resolution data is typically at 'scale0'
morph_full = morph['scale0'].to_dataset()

In [72]:
print(f"Spatial dimensions: y={morph_full.dims['y']}, x={morph_full.dims['x']}")

Spatial dimensions: y=20495, x=19973


/var/folders/w5/x66nm95j3cx1k13smkb3sky00000gn/T/ipykernel_34528/985696611.py:1: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Spatial dimensions: y={morph_full.dims['y']}, x={morph_full.dims['x']}")


In [ ]:
# The actual array is accessed via the data variable (usually called something like 'image' or the dataset name)
# Let's find the data variable name:
print(f"\nData variables: {list(morph_full.data_vars)}")


Data variables: ['image']


In [94]:
morph_array = morph_full['image']

# Extract relevant channels in scale0 which is the highest resolution we have
# Adjust indices based on actual channel order in your file
alphasma_vim = morph_array.sel(c='AlphaSMA/Vimentin').values  # Vessel marker
atp1a1 = morph_array.sel(c='ATP1A1/CD45/E-Cadherin').values  # Membrane marker
dapi = morph_array.sel(c='DAPI').values  # Nuclei marker

In [95]:
viewer = napari.Viewer()

viewer.add_image(
    atp1a1,
    name='ATP1A1 (membranes)',
    colormap='green',
    scale=[0.2125, 0.2125],
    blending='additive',
    visible=True
)

viewer.add_image(
    alphasma_vim,
    name='AlphaSMA/Vimentin (vessels)',
    colormap='magenta',
    scale=[0.2125, 0.2125],
    blending='additive',
    visible=True
)

viewer.add_image(
    dapi,
    name='DAPI (nuclei)',
    colormap='blue',
    scale=[0.2125, 0.2125],  # Xenium pixel size
    blending='additive',
    visible=True
)

/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (20495, 19973) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (20495, 19973) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(
/Users/lennonmccartney/envs/xenium_scverse_py310/lib/python3.10/site-packages/napari/_vispy/layers/scalar_field.py:197: UserWarning: data shape (20495, 19973) exceeds GL_MAX_TEXTURE_SIZE 16384 in at least one axis and will be downsampled. Rendering is currently in 2D mode.
  warnings.warn(


<Image layer 'DAPI (nuclei)' at 0x3c1420be0>